# Bank Marketing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler # To handle imbalanced data
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Load and Explore the Dataset

In [ ]:
data = pd.read_csv('/content/bank-full.csv', sep=';')



data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/bank-full.csv'

In [ ]:
data.info()

In [ ]:
data.describe()

# Data Cleaning and Preprocessing

-  Hnadle missing values

In [ ]:

data.isnull().sum()
# No missing values

- Convert catogrical features

In [ ]:
data['y'] = data['y'].map({'yes': 1, 'no': 0})
data['y'].head()

- Encodeing

In [ ]:
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
data_encoded = pd.get_dummies(data, columns=categorical_features)

# when i have tried to delete the categorical_features the accuracy decreasing
# data_encoded = data.drop(categorical_features, axis=1)

In [ ]:
data_encoded.info()

- class distribution

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='y', data=data_encoded)
plt.title('Class Distribution')

- We have imbalnced data so that will impact on modle performance

- There are more than solutions , I will drop some samples of class 0 to be balanced with 1

In [ ]:
# X = data_encoded.drop('y', axis=1)
# y = data_encoded['y']


# rus = RandomUnderSampler(random_state=42)
# X_resampled, y_resampled = rus.fit_resample(X, y )






# # New dataFrame

# data_balanced = pd.concat([X_resampled, y_resampled], axis=1)





In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='y', data=data_balanced)

# Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Apply Logistic Regression

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)


- Model prediction

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
correct_predictions = (y_pred == y_test).sum()
total_predictions = len(y_test)

accuracy = (correct_predictions / total_predictions) * 100
print(f"Modle predict {correct_predictions} correct out of {total_predictions}")
print("Accuracy:", np.round(accuracy) )

# Apply a Neural Network

- 1. Convert numpy arrays to PyTorch tensors (Important for PyTorch models)

- Tensors are used to speed up the computaions

In [ ]:
X_train_tensor = torch.tensor(X_train.values.astype(np.int32), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)


X_test_tensor = torch.tensor(X_test.values.astype(np.int32), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)
#


# 2. Create DataLoader for batch processing
# Using DataLoader is essential for efficient NN training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Determine input dimension (number of features)
INPUT_DIM = X_train.shape[1]

- Define the Neural Network

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32) # (input , output)
        self.layer3 = nn.Linear(32, 16)
        self.output_layer = nn.Linear(16, 1)
        self.activation1 = nn.ReLU()
        # We will apply sigmoind in loss function



    def forward(slef , x):
      x = slef.layer1(x)
      x = slef.activation1(x)


      # PASS TO NEXT LAYER
      x = slef.layer2(x)
      x = slef.activation1(x)


      x = slef.layer3(x)
      x = slef.activation1(x)

      return slef.output_layer(x)





model = NeuralNetwork(input_dim=INPUT_DIM)

- Loss, Optimizer, and Training Loop

In [ ]:

# Loss Function (for binary classification)
criterion = nn.BCEWithLogitsLoss()
# Optimizer (Adam is a good default choice)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training Parameters
NUM_EPOCHS = 10

# Training Loop
for epoch in range(NUM_EPOCHS):
    for X_batch, y_batch in train_loader:
        # 1. Zero the gradients
        optimizer.zero_grad()
        # 2. Forward pass: compute predicted outputs
        y_pred_logits = model(X_batch)

        # 3. Compute the loss
        loss = criterion(y_pred_logits, y_batch)

        # 4. Backward pass: compute gradient of the loss
        loss.backward()

        # 5. Update model parameters
        optimizer.step()


In [ ]:
# Set model to evaluation mode (turns off features like Dropout if they exist)
model.eval()

with torch.no_grad(): # Disable gradient calculations during inference
    # Forward pass on the test data to get logits
    test_logits = model(X_test_tensor)

    # Convert logits to probabilities using Sigmoid
    y_prob_nn = torch.sigmoid(test_logits)

    # Convert probabilities to binary predictions (0 or 1)
    y_pred_nn = (y_prob_nn > 0.5).int()

# Calculate Accuracy (convert tensors back to numpy for easy calculation)
y_pred_numpy = y_pred_nn.numpy().flatten()
y_test_numpy = y_test_tensor.numpy().flatten()

correct_predictions_nn = (y_pred_numpy == y_test_numpy).sum()
total_predictions_nn = len(y_test_numpy)

accuracy_nn = (correct_predictions_nn / total_predictions_nn) * 100

print(f"NN Modle predict {correct_predictions_nn} correct out of {total_predictions_nn}")
print("NN Accuracy:", np.round(accuracy_nn, 2))

# Random Forest Classifier (Scikit-learn)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the Model
# n_estimators is the number of trees in the forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the Model
# Use your original NumPy arrays (X_train, y_train)
rf_model.fit(X_train, y_train)

# 3. Predict and Evaluate
y_pred_rf = rf_model.predict(X_test)

correct_predictions_rf = (y_pred_rf == y_test).sum()
total_predictions_rf = len(y_test)

accuracy_rf = (correct_predictions_rf / total_predictions_rf) * 100

print(f"Random Forest Modle predict {correct_predictions_rf} correct out of {total_predictions_rf}")
print("Random Forest Accuracy:", np.round(accuracy_rf, 2))

# Gradient Boosting (e.g., XGBoost)

In [ ]:
# You would need to install this library first: pip install xgboost
import xgboost as xgb

# 1. Initialize the Model
# Parameters are typically tuned for performance
xgb_model = xgb.XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss', random_state=42)

# 2. Train the Model
xgb_model.fit(X_train, y_train)

# 3. Predict and Evaluate
y_pred_xgb = xgb_model.predict(X_test)

correct_predictions_xgb = (y_pred_xgb == y_test).sum()
total_predictions_xgb = len(y_test)

accuracy_xgb = (correct_predictions_xgb / total_predictions_xgb) * 100

print(f"XGBoost Modle predict {correct_predictions_xgb} correct out of {total_predictions_xgb}")
print("XGBoost Accuracy:", np.round(accuracy_xgb, 2))